# 02 — Limpieza, split temporal y features
## Proyecto final de Machine Learning — Grupo 11 · Lending Club

**Objetivo:** dejar listo el conjunto de datos para modelar, con tres decisiones documentadas:
1. **Split temporal** confiable (evitando censura por la derecha).
2. **Limpieza** por reglas de dominio y tratamiento de outliers sin filtrar información del futuro.
3. **Ingeniería de variables** y preprocesador de sklearn ajustado **solo con train**.

La lógica reutilizable vive en `src/data.py`, `src/features.py` y `src/config.py`; este notebook la usa y explica.


In [ ]:
import sys
from pathlib import Path

# El notebook vive en notebooks/; la raíz del repo es el directorio padre
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config, data, features

sns.set_theme(style="whitegrid")
np.random.seed(config.SEED)
pd.set_option("display.max_columns", 100)


## 1. Carga de datos

In [ ]:
df_raw = data.load_raw()
print(f"Filas: {len(df_raw):,} | Columnas: {df_raw.shape[1]}")


## 2. Diagnóstico de censura: ¿qué años son confiables?

Un préstamo solo entra al modelo si ya tiene desenlace definitivo (`Fully Paid`, `Charged Off`, `Default`).
Los préstamos recientes (y los de 60 meses) todavía están `Current` en muchos casos, y los que **sí** tienen desenlace
no son una muestra representativa: son los que se pagaron antes de tiempo o cayeron en default rápido.
Esto se llama *censura por la derecha* y sesga la tasa de default hacia abajo o hacia arriba según el año.

La tabla muestra, por año de originación y plazo, cuántos préstamos hay, qué porcentaje ya está resuelto
y la tasa de default entre los resueltos. **Esta tabla se calcula sobre el dataset completo solo para diseñar el split;
no se usa para elegir modelos.**


In [ ]:
tabla = data.censoring_table(df_raw)
tabla


In [ ]:
t = tabla.reset_index()
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.lineplot(data=t, x="issue_year", y="pct_resueltos", hue="term", marker="o", ax=axes[0])
axes[0].set_title("% de préstamos con desenlace definitivo")
axes[0].set_ylabel("proporción resuelta")
sns.lineplot(data=t, x="issue_year", y="tasa_default_resueltos", hue="term", marker="o", ax=axes[1])
axes[1].set_title("Tasa de default entre resueltos")
axes[1].set_ylabel("tasa de default")
plt.tight_layout()
plt.show()


### Decisión del split

Criterio: usar años en los que el porcentaje resuelto sea alto y la tasa de default se vea estable entre plazos.
Los valores vigentes están en `src/config.py`:

| Conjunto | Años de originación | Uso |
|---|---|---|
| Train | `TRAIN_YEARS` (2012–2014) | ajustar modelos y preprocesador |
| Validación | `VALID_YEAR` (2015) | elegir modelo e hiperparámetros |
| Test | `TEST_YEAR` (2016) | **una sola** evaluación final |

> **Completar tras ver la tabla de arriba:** confirmar (o ajustar en `config.py`) estos años y, si el plazo de 60 meses
> sigue muy incompleto en 2015–2016, decidir si se fija `ONLY_TERM = " 36 months"`. Documentar aquí la justificación
> con los números reales de la tabla.


## 3. Target, limpieza por reglas de dominio y variables derivadas

In [ ]:
df = data.build_target(df_raw)
df = data.add_issue_year(df)
print(f"Filas con desenlace definitivo: {len(df):,} | tasa de default global: {df['target'].mean():.3f}")

df, reglas = features.clean(df, report=True)
pd.Series(reglas, name="filas afectadas (pasan a NaN)").to_frame()


Las reglas (`dti` fuera de [0, 100], `revol_util` > 150 %, `annual_inc` ≤ 0) **no eliminan filas**:
el valor imposible pasa a faltante y se imputa después con la mediana de *train*. Así no se pierde el préstamo
y no se filtra información de otros conjuntos.


In [ ]:
df = features.engineer(df)
nuevas = ["term_months", "emp_length_years", "fico_avg", "credit_history_years", "loan_to_income", "purpose_grouped"]
df[nuevas].describe(include="all").T


Variables derivadas: `term_months` y `emp_length_years` (texto → número), `fico_avg` (promedio del rango FICO, evita redundancia),
`credit_history_years` (antigüedad del historial al momento de la solicitud), `loan_to_income` (monto / ingreso anual)
y `purpose_grouped` (3 categorías). Todas usan solo información disponible al solicitar el préstamo.


## 4. Split temporal

In [ ]:
train, valid, test = data.temporal_split(df)
data.split_summary(train, valid, test)


La tasa de default del test no se muestra a propósito: no se deben tomar decisiones de modelado mirando el conjunto de evaluación final.


## 5. Faltantes por año (después de la limpieza)

In [ ]:
cols_na = ["mort_acc", "emp_length_years", "revol_util", "credit_history_years", "dti", "loan_to_income"]
faltantes_anio = train.groupby("issue_year")[cols_na].agg(lambda s: s.isna().mean() * 100)
fig, ax = plt.subplots(figsize=(8, 3.5))
sns.heatmap(faltantes_anio.T, annot=True, fmt=".1f", cmap="Reds", cbar_kws={"label": "% faltante"}, ax=ax)
ax.set_title("% de valores faltantes por año de originación (train)")
plt.tight_layout()
plt.show()


Si un campo tiene faltantes concentrados en ciertos años (por ejemplo, variables que Lending Club empezó a capturar más tarde),
el faltante es **informativo** y no aleatorio. Por eso el preprocesador agrega indicadores de faltante (`add_indicator=True`).


## 6. Outliers: winsorización ajustada solo con train

In [ ]:
cols = ["annual_inc", "revol_bal", "loan_to_income"]
w = features.Winsorizer(columns=cols).fit(train)
antes = train[cols].describe(percentiles=[.5, .99, .995]).T[["50%", "99%", "99.5%", "max"]]
despues = w.transform(train[cols]).describe().T[["max"]].rename(columns={"max": "max (recortado)"})
pd.concat([antes, despues], axis=1)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.boxplot(y=train["annual_inc"], ax=axes[0]); axes[0].set_title("annual_inc (original)")
sns.boxplot(y=w.transform(train[cols])["annual_inc"], ax=axes[1]); axes[1].set_title("annual_inc (winsorizado)")
plt.tight_layout()
plt.show()


Los cuantiles de recorte (0.5 % y 99.5 %) se aprenden en train y se aplican igual a validación y test.


## 7. Señal de las variables nuevas (solo train)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["credit_history_years", "loan_to_income", "emp_length_years"]):
    sns.kdeplot(data=train, x=col, hue="target", common_norm=False, ax=ax, warn_singular=False)
    ax.set_title(col)
plt.tight_layout()
plt.show()


## 8. Estabilidad entre conjuntos (solo features, sin mirar el target del test)

In [ ]:
num, cat = features.get_feature_lists()
comparacion = pd.DataFrame({
    "train": train[num].mean(),
    "valid": valid[num].mean(),
    "test": test[num].mean(),
})
comparacion["dif. valid vs train (%)"] = ((comparacion["valid"] / comparacion["train"]) - 1) * 100
comparacion.round(3)


Diferencias grandes en una variable entre train y test son señal de *drift* en los datos de entrada. Se documentan como limitación.


## 9. Preprocesador (ajustado solo con train)

In [ ]:
prep = features.build_preprocessor(include_grey_zone=False)
X_train = prep.fit_transform(train[num + cat])   # fit SOLO con train
X_valid = prep.transform(valid[num + cat])
X_test = prep.transform(test[num + cat])

print("Shapes:", X_train.shape, X_valid.shape, X_test.shape)
print("NaN restantes en valid:", int(np.isnan(X_valid).sum()))
print("Nº de features finales:", len(prep.get_feature_names_out()))
list(prep.get_feature_names_out())[:15]


## 10. Guardar los splits procesados

In [ ]:
config.DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
for nombre, d in {"train": train, "valid": valid, "test": test}.items():
    d.to_parquet(config.DATA_PROCESSED / f"{nombre}.parquet", index=False)
print("Guardado en", config.DATA_PROCESSED)


## 11. Decisiones y conclusiones (completar con los resultados reales)

- **Split:** años elegidos y justificación según la tabla de censura.
- **Limpieza:** número de filas afectadas por cada regla (sección 3).
- **Outliers:** winsorización 0.5 % / 99.5 % ajustada en train.
- **Faltantes:** imputación por mediana/moda con indicadores de faltante; comentar si hay patrón por año.
- **Variables derivadas:** `fico_avg`, `credit_history_years`, `loan_to_income`, `term_months`, `emp_length_years`, `purpose_grouped`.
- **Descartadas:** `addr_region` (sin mejora en el baseline), `emp_title` (texto libre), columnas joint/`sec_app_*` (alta tasa de faltantes), variables con leakage (ver `proposal.md`).
- **Siguiente paso:** `03_modelos.ipynb` — baseline re-ejecutado con este split y comparación de al menos 3 familias de modelos.
